In [ ]:
import pandas as pd
import statsmodels.api as sm
from tabulate import tabulate
import os
import matplotlib.pyplot as plt
import numpy as np

# List of games to analyze
games = [
    {"name": "Palworld", "file": "../data:clean/palworld_2024-12-09_to_2025-01-13.csv", "update": "2024-12-23"},
    {"name": "Warframe", "file": "../data:clean/warframe_2025-11-26_to_2025-12-31.csv", "update": "2025-12-10"},
    {"name": "Counter-Strike 2", "file": "../data:clean/counter_strike_2_2024-04-23_to_2024-05-28.csv", "update": "2024-05-07"},
    {"name": "Sea of Thieves", "file": "../data:clean/sea_of_thieves_2024-10-03_to_2024-11-07.csv", "update": "2024-10-17"},
    {"name": "PUBG: BATTLEGROUNDS", "file": "../data:clean/pubg__battlegrounds_2025-10-22_to_2025-11-26.csv", "update": "2025-11-05"},
    {"name": "Dead by Daylight", "file": "../data:clean/dead_by_daylight_2023-11-14_to_2023-12-19.csv", "update": "2023-11-28"},
    {"name": "Don't Starve Together", "file": "../data:clean/don_t_starve_together_2023-04-13_to_2023-05-18.csv", "update": "2023-04-27"},
    {"name": "Deep Rock Galactic", "file": "../data:clean/deep_rock_galactic_2024-05-30_to_2024-07-04.csv", "update": "2024-06-13"},
    {"name": "Apex Legends", "file": "../data:clean/apex_legends_2025-01-28_to_2025-03-04.csv", "update": "2025-02-11"},
    {"name": "Destiny 2", "file": "../data:clean/destiny_2_2024-09-24_to_2024-10-29.csv", "update": "2024-10-08"},
    {"name": "No Man's Sky", "file": "../data:clean/no_man_s_sky_2025-01-15_to_2025-02-19.csv", "update": "2025-01-29"},
    {"name": "HELLDIVERS 2", "file": "../data:clean/helldivers_2_2025-08-19_to_2025-09-23.csv", "update": "2025-09-02"}
]

all_games_df = pd.DataFrame()

for game in games:
    df = pd.read_csv(game['file'])
    df['DateTime'] = pd.to_datetime(df['DateTime'])
    update_date = pd.to_datetime(game['update'])
    df['game'] = game['name']
    df['days_from_update'] = (df['DateTime'] - update_date).dt.days
    all_games_df = pd.concat([all_games_df, df])

all_games_df['Players_scaled'] = all_games_df.groupby('game')['Players'].transform(lambda x: (x - x.mean()) / x.std())
all_games_df = all_games_df.dropna(subset=['Players_scaled'])

# --- 1. Event Study / Dynamic Effects --- #
event_window = 14 # days before and after
filtered_df = all_games_df[all_games_df['days_from_update'].abs() <= event_window].copy()

# Create dummy variables for each day relative to the update
# We omit one day (e.g., day before update, -1) as the reference category
event_dummies = pd.get_dummies(filtered_df['days_from_update'], prefix='day')
event_dummies = event_dummies.drop('day_-1', axis=1) # Drop reference category

y = filtered_df['Players_scaled']
X = sm.add_constant(event_dummies)

event_model = sm.OLS(y, X).fit()

# Prepare data for plotting
coeffs = event_model.params.drop('const')
conf_int = event_model.conf_int().drop('const')
days = [int(c.replace('day_', '')) for c in coeffs.index]

# Add the reference day back in with a coefficient of 0
plot_data = pd.DataFrame({
    'day': days,
    'coeff': coeffs.values,
    'conf_low': conf_int[0],
    'conf_high': conf_int[1]
}).sort_values('day').reset_index(drop=True)

plot_data = pd.concat([plot_data, pd.DataFrame({'day': [-1], 'coeff': [0], 'conf_low': [0], 'conf_high': [0]})], ignore_index=True).sort_values('day')

# Plotting the dynamic effects
plt.figure(figsize=(12, 7))
plt.errorbar(plot_data['day'], plot_data['coeff'], yerr=[plot_data['coeff'] - plot_data['conf_low'], plot_data['conf_high'] - plot_data['coeff']], fmt='-o', capsize=5, label='Coefficient')
plt.axhline(0, color='red', linestyle='--')
plt.axvline(0, color='grey', linestyle='--', label='Update Day')
plt.title('Event Study: Dynamic Effect of Updates on Player Count')
plt.xlabel('Days Relative to Update')
plt.ylabel('Coefficient (Change in Std. Deviations of Players)')
plt.grid(True, alpha=0.3)
plt.legend()

output_dir = '../outputs'
os.makedirs(output_dir, exist_ok=True)
plot_path = os.path.join(output_dir, 'dynamic_effects_plot.png')
plt.savefig(plot_path)
print(f'Dynamic effects plot saved to {plot_path}')

# --- 2. Interaction Effects --- #
# Categorize games as primarily Co-op or PvP
coop_games = ['Palworld', 'Warframe', 'Sea of Thieves', "Don't Starve Together", 'Deep Rock Galactic', 'Destiny 2', 'No Man\'s Sky', 'HELLDIVERS 2']
all_games_df['is_coop'] = all_games_df['game'].isin(coop_games).astype(int)
all_games_df['post_update'] = (all_games_df['days_from_update'] >= 0).astype(int)

# Create the interaction term
all_games_df['interaction'] = all_games_df['post_update'] * all_games_df['is_coop']

y_interact = all_games_df['Players_scaled']
X_interact = all_games_df[['post_update', 'is_coop', 'interaction']]
X_interact = sm.add_constant(X_interact)

interaction_model = sm.OLS(y_interact, X_interact).fit()

# --- Generate and Save Tables --- #
def create_regression_table(model, model_name):
    return model.summary().as_text()

event_table = event_model.summary().as_html()
interaction_table = interaction_model.summary().as_html()

with open(os.path.join(output_dir, 'event_study_table.html'), 'w') as f:
    f.write(event_table)
print('Event study regression table saved.')

with open(os.path.join(output_dir, 'interaction_effects_table.html'), 'w') as f:
    f.write(interaction_table)
print('Interaction effects regression table saved.')

print('\n--- Interaction Model Summary ---')
print(interaction_model.summary())